# DisasterM3 — U-Net Segmentation Fine-Tuning (Stage 1 of Hybrid Triage)
**Purpose:** Train a ResNet34-backed U-Net to produce 1024×1024 damage masks from the
37,204 Referring Expression Segmentation entries that were *excluded* from the VLM
fine-tuning pipeline (D6: mask-path targets are unusable for text generation).

**Hardware:** Kaggle T4 GPU (16 GB VRAM) — U-Net is lightweight (~21M params), so
no quantization needed.

**Architecture mirrors `train_qwen_disasterm3.ipynb`:**
| Pattern | VLM notebook | This notebook |
|---|---|---|
| Pinned deps + idempotent install | Cell 1 | Cell 2 |
| All config in one cell | Cell 2 | Cell 3 |
| Manifest audit + data prep | Cell 4 | Cell 5 |
| Checkpoint-resume across sessions | Cells 8.5, 9 | Cell 8 |
| TimeLimitCallback (12 h cap) | Cell 10 | Cell 9 |
| HF Hub push on session end | Cell 10b | Cell 11 |

### How to run ("Run All" workflow)
1. **Prerequisites (once):** GPU T4, internet ON, DisasterM3 mirror dataset attached,
   `HF_TOKEN` secret (write-scoped) for checkpoint backup.
2. **Fresh session:** Run All → Cell 2 installs deps and HALTS → Restart kernel →
   Run All again. Cell 2 skips installation and training proceeds.
3. Training auto-saves checkpoints every N epochs; Cell 11 pushes to HF Hub.
4. **Resume session:** Set `RESUME_EPOCH` and `HF_CHECKPOINT_REPO` in Cell 3, then Run All.

### Relationship to VLM pipeline
This notebook trains the **spatial extraction** stage of the Hybrid Triage Pipeline.
The U-Net produces masks → `3_vllm_evaluation.ipynb` counts buildings mathematically, 
while extracting bounding boxes so the fine-tuned Qwen2.5-VL reasons about the cropped patches.

In [1]:
# ── Cell 1: Environment Setup (Run-All safe) ──────────────────────────────
# Mirrors train_qwen_disasterm3.ipynb Cell 1: pinned versions, idempotent,
# halts on first install to force kernel restart.
import importlib.metadata as _md

PINS = {
    "torch": "2.4.1",
    "torchvision": "0.19.1",
    "torchaudio": "2.4.1",
    "segmentation-models-pytorch": "0.3.4",
    "albumentations": "1.4.21",
    "opencv-python-headless": "4.10.0.84",
}

_mismatched = []
for _pkg, _want in PINS.items():
    try:
        _have = _md.version(_pkg)
    except _md.PackageNotFoundError:
        _have = "not installed"
    if _have != _want:
        _mismatched.append(f"{_pkg}: {_have} → {_want}")

if _mismatched:
    print("⏳ Installing pinned versions:")
    for _m in _mismatched:
        print(f"   {_m}")
    !pip install -q \
        torch==2.4.1 \
        torchvision==0.19.1 \
        torchaudio==2.4.1 \
        segmentation-models-pytorch==0.3.4 \
        albumentations==1.4.21 \
        opencv-python-headless==4.10.0.84 \
        pillow \
        huggingface_hub[hf_xet]
    raise SystemExit(
        "✓ Dependencies installed. RESTART THE KERNEL NOW "
        "(Run → Restart & clear cell outputs), then click 'Run All' again — "
        "this cell will detect the correct versions and skip installation."
    )

print("✓ All pinned versions already installed — proceeding.")

✓ All pinned versions already installed — proceeding.


In [2]:
# ── Cell 2: Configuration ─────────────────────────────────────────────────
# Mirrors train_qwen_disasterm3.ipynb Cell 2: all hyperparameters in one place.
import os
import json
import torch
from pathlib import Path
from datetime import datetime

# ── Paths (same DisasterM3 mirror dataset mount as VLM notebook) ──
DATA_ROOT = Path("/kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct")
MANIFEST_PATH = DATA_ROOT / "train_release.json"

# ── Model ──
# Using segmentation_models_pytorch (smp) for a proper U-Net with skip connections,
# which is strictly superior to Aryan's sequential decoder (no skip connections).
ENCODER_NAME = "resnet34"     # ImageNet-pretrained backbone
ENCODER_WEIGHTS = "imagenet"  # Transfer learning
NUM_CLASSES = 4               # 0=Background, 1=Intact, 2=Damaged

# ── Training config ──
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-4
NUM_EPOCHS = 40              
BATCH_SIZE = 16                 # Reduced from 16 to fit in 15GB VRAM
NUM_WORKERS = 2                # Reduced to prevent CPU bottleneck
IMAGE_SIZE = 512               # Resize from 1024 to save VRAM/time; still high-res

# ── Checkpointing (12-hour Kaggle session cap) ──
# Mirrors VLM notebook: save frequently, push to HF, resume next session.
CHECKPOINT_DIR = "/kaggle/working/unet_checkpoints"
SAVE_EVERY_N_EPOCHS = 2       # Save checkpoint every 2 epochs
TIME_LIMIT_HOURS = 11.4       # Same as VLM notebook

# ── Cross-session resume (mirrors VLM notebook Cell 8.5) ──
# After first session: push checkpoint to HF, set this to the repo ID.
HF_CHECKPOINT_REPO = None     # e.g. "AbrarAlam/disasterm3-unet-checkpoints"
RESUME_EPOCH = 0              # Set to the epoch number to resume from

# ── Output ──
OUTPUT_DIR = "/kaggle/working/unet_disasterm3"
MODEL_NAME = f"disasterm3_unet_{ENCODER_NAME}_ep{NUM_EPOCHS}"

print(f"✓ Config loaded")
print(f"  Encoder: {ENCODER_NAME} (pretrained={ENCODER_WEIGHTS})")
print(f"  Classes: {NUM_CLASSES} (Background, Intact, Damaged)")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Image size: {IMAGE_SIZE}×{IMAGE_SIZE}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

✓ Config loaded
  Encoder: resnet34 (pretrained=imagenet)
  Classes: 4 (Background, Intact, Damaged)
  Batch size: 16
  Image size: 512×512
  Epochs: 40
  Device: Tesla T4
  VRAM: 15.6 GB


## 2. Load & Prepare Segmentation Data
We use the **37,204 Referring Expression Segmentation entries** that were excluded
from the VLM fine-tuning (D6). Their `ground_truth` field contains the mask file
path, and the images are the same pre/post-disaster pairs from the DisasterM3 dataset.

In [4]:
# ── Cell 4: Load manifest and extract segmentation entries ────────────────
# Mirrors train_qwen_disasterm3.ipynb Cell 4: same manifest, same path
# resolution logic, but this time we KEEP the segmentation entries and
# DROP the text-QA entries (exact inverse of the VLM notebook).
# ── Cell 4 (rewritten): Build combined multi-class masks from 3 binary sources ──
import cv2
import numpy as np
import os
from collections import Counter, defaultdict
from pathlib import Path
import json

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)
print(f"Loaded {len(raw_data):,} entries from manifest")

# ── Filter: ONLY Building Damage Assessment segmentation entries ──
bda_entries = [
    e for e in raw_data
    if e.get("task") == "Referring Expression Segmentation"
    and e.get("cls_description") == "Building Damage Assessment"
]
print(f"Building Damage Assessment entries: {len(bda_entries):,}")

# ── Path resolution (same logic as before) ──
def resolve_path(rel_path):
    if not rel_path:
        return None
    rel_path = rel_path.replace("\\", "/")
    filename = Path(rel_path).name
    candidates = [
        DATA_ROOT / rel_path,
        DATA_ROOT / "train_images" / rel_path,
        DATA_ROOT / "train_images" / "train_images" / filename,
        DATA_ROOT / "masks" / rel_path,
        DATA_ROOT / "masks" / "masks" / filename,
    ]
    for c in candidates:
        if c.exists():
            return str(c)
    return None

# ── Group entries by underlying post-disaster image ──
by_image = defaultdict(dict)  # post_image_path -> {folder_type: mask_path}

for e in bda_entries:
    post_img = e.get("post_image_path", "")
    if e.get("image_type") != "Optical":   # skip SAR duplicates, keep optical only
        continue
    mask_rel = e.get("ground_truth", "")
    folder = Path(mask_rel.replace("\\", "/")).parent.name  # e.g. train_building_intact_mask
    resolved = resolve_path(mask_rel)
    if resolved:
        by_image[post_img][folder] = resolved
        by_image[post_img]["post_image_path"] = post_img

print(f"Unique images with at least one mask: {len(by_image):,}")

# ── Build combined (image, mask) pairs ──
FOLDER_TO_CLASS = {
    "train_building_intact_mask": 1,
    "train_building_damaged_mask": 2,
    "train_building_destroyed_mask": 3,   # use 2 here instead if you want to collapse into "damaged"
}

# ── FIX: Create temporary directory on Kaggle hard drive for masks ──
os.makedirs("/tmp/combined_masks", exist_ok=True)

pairs = []
skipped_img = 0

for post_img, mask_dict in by_image.items():
    img_path = resolve_path(post_img)
    if not img_path:
        skipped_img += 1
        continue

    combined_mask = None
    for folder, class_id in FOLDER_TO_CLASS.items():
        mask_path = mask_dict.get(folder)
        if mask_path is None:
            continue
        m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if m is None:
            continue
        if combined_mask is None:
            combined_mask = np.zeros_like(m, dtype=np.uint8)
        combined_mask[m > 0] = class_id   # later classes overwrite earlier if they overlap

    if combined_mask is None:
        continue  # no valid masks found for this image at all

    # ── FIX: Save mask to hard drive instead of RAM ──
    mask_filename = Path(img_path).name
    save_path = f"/tmp/combined_masks/{mask_filename}"
    cv2.imwrite(save_path, combined_mask)

    pairs.append({
        "image_path": img_path,
        "mask_path": save_path,   # Store the string path, NOT the array!
    })

print(f"\n✓ Built {len(pairs):,} valid (image, mask) pairs")
print(f"  Skipped (image not found): {skipped_img:,}")

# Sanity check
if pairs:
    print(f"\n  Sample: {pairs[0]['image_path']}")
    print(f"  Mask saved at: {pairs[0]['mask_path']}")


Loaded 92,968 entries from manifest
Building Damage Assessment entries: 14,531
Unique images with at least one mask: 6,443

✓ Built 6,443 valid (image, mask) pairs
  Skipped (image not found): 0

  Sample: /kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct/train_images/train_images/bata_explosion_post_0.png
  Mask saved at: /tmp/combined_masks/bata_explosion_post_0.png


In [5]:
# ── Cell 5: PyTorch Dataset with augmentations ────────────────────────────
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class DisasterM3SegDataset(Dataset):
    """Dataset for DisasterM3 segmentation entries.
    
    Loads post-disaster images and their pre-merged combined damage masks.
    Masks were already built as multi-class in Cell 4 from the 3 binary
    source masks (intact / damaged / destroyed), so no further label
    simplification is needed here:
      0 = Background
      1 = Intact (no damage)
      2 = Damaged
      3 = Destroyed
    """
    
    def __init__(self, pairs, transform=None, image_size=512):
        self.pairs = pairs
        self.transform = transform
        self.image_size = image_size
    
    def __len__(self):
        return len(self.pairs)
    
    def __getitem__(self, idx):
        pair = self.pairs[idx]
        
        # Load image
        image = cv2.imread(pair["image_path"])
        if image is None:
            return self.__getitem__((idx + 1) % len(self))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # FIX: Load mask from the hard drive path we generated
        mask = cv2.imread(pair["mask_path"], cv2.IMREAD_GRAYSCALE)
        if mask is None:
            return self.__getitem__((idx + 1) % len(self))
        
        # Apply augmentations
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']
            
        return image, mask.long()



# ── Augmentation pipelines ──
train_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

# ── Train/Val split (90/10) ──
from sklearn.model_selection import train_test_split

train_pairs, val_pairs = train_test_split(pairs, test_size=0.1, random_state=42)

train_dataset = DisasterM3SegDataset(train_pairs, transform=train_transform, image_size=IMAGE_SIZE)
val_dataset = DisasterM3SegDataset(val_pairs, transform=val_transform, image_size=IMAGE_SIZE)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"✓ Datasets created")
print(f"  Train: {len(train_dataset):,} samples ({len(train_loader):,} batches)")
print(f"  Val:   {len(val_dataset):,} samples ({len(val_loader):,} batches)")

/usr/local/lib/python3.12/dist-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.21). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


✓ Datasets created
  Train: 5,798 samples (363 batches)
  Val:   645 samples (41 batches)


In [6]:
import segmentation_models_pytorch as smp
import torch.nn as nn

NUM_CLASSES = 4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class PhasedCompoundLoss(nn.Module):
    def __init__(self, num_epochs, lovasz_start_fraction=0.8):
        super().__init__()
        self.tversky = smp.losses.TverskyLoss(
            mode='multiclass', alpha=0.3, beta=0.7, gamma=2.0,
        )
        self.lovasz = smp.losses.LovaszLoss(mode='multiclass')
        self.lovasz_start_epoch = int(num_epochs * lovasz_start_fraction)
        self.current_epoch = 0

    def set_epoch(self, epoch):
        self.current_epoch = epoch

    def forward(self, logits, targets):
        t_loss = self.tversky(logits, targets)
        if self.current_epoch >= self.lovasz_start_epoch:
            l_loss = self.lovasz(logits, targets)
            total = 0.6 * t_loss + 0.4 * l_loss
            return total, t_loss.item(), l_loss.item()
        else:
            return t_loss, t_loss.item(), 0.0

criterion = PhasedCompoundLoss(num_epochs=NUM_EPOCHS).to(DEVICE)
print(f"✓ Phased Compound Loss ready on {DEVICE}")

✓ Phased Compound Loss ready on cuda


In [7]:
# ── Cell 7: Resume from HF checkpoint (mirrors VLM notebook Cell 8.5) ────
# Cross-session resume: download the latest checkpoint from HF Hub.
start_epoch = 0

if HF_CHECKPOINT_REPO:
    from huggingface_hub import hf_hub_download
    from kaggle_secrets import UserSecretsClient
    
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    
    try:
        ckpt_path = hf_hub_download(
            repo_id=HF_CHECKPOINT_REPO,
            filename=f"unet_epoch_{RESUME_EPOCH}.pth",
            token=hf_token,
        )
        checkpoint = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
        start_epoch = checkpoint["epoch"] + 1
        print(f"✓ Resumed from HF checkpoint: epoch {checkpoint['epoch']}")
        print(f"  Starting at epoch {start_epoch}")
    except Exception as e:
        print(f"⚠ Could not load HF checkpoint: {e}")
        print("  Starting fresh.")
else:
    # Check local checkpoint directory (same session resume)
    if os.path.exists(CHECKPOINT_DIR):
        ckpts = sorted(
            [f for f in os.listdir(CHECKPOINT_DIR) if f.endswith(".pth")],
            key=lambda x: int(x.split("_")[-1].split(".")[0])
        )
        if ckpts:
            latest = os.path.join(CHECKPOINT_DIR, ckpts[-1])
            checkpoint = torch.load(latest, map_location=device)
            model.load_state_dict(checkpoint["model_state_dict"])
            optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
            scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
            start_epoch = checkpoint["epoch"] + 1
            print(f"✓ Resumed from local checkpoint: {latest}")
            print(f"  Starting at epoch {start_epoch}")
        else:
            print("No checkpoints found — starting fresh.")
    else:
        print("No checkpoint directory — starting fresh.")

No checkpoint directory — starting fresh.


In [8]:
import time
import os
import datetime
import json
import numpy as np
import segmentation_models_pytorch as smp
from tqdm import tqdm
from IPython.display import display, HTML
from torch.cuda.amp import GradScaler, autocast

training_start = time.time()
epoch_times = []

# ── Model ──
model = smp.Unet(
    encoder_name="efficientnet-b4",
    encoder_weights="imagenet",
    in_channels=3,
    classes=NUM_CLASSES,
).to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
scaler = GradScaler() # AMP Optimization

def compute_iou(pred, target, num_classes):
    ious = []
    for cls in range(num_classes):
        pred_cls = (pred == cls)
        target_cls = (target == cls)
        intersection = (pred_cls & target_cls).sum().item()
        union = (pred_cls | target_cls).sum().item()
        if union == 0:
            ious.append(float('nan'))
        else:
            ious.append(intersection / union)
    return ious

def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    total_loss = 0.0
    for images, masks in tqdm(loader, desc="Train", leave=False):
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        
        # Mixed Precision Forward Pass
        with autocast():
            logits = model(images)
            loss, ce_val, dice_val = criterion(logits, masks)
            
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_ious = [[] for _ in range(NUM_CLASSES)]
    
    for images, masks in tqdm(loader, desc="Val", leave=False):
        images, masks = images.to(device), masks.to(device)
        
        with autocast():
            logits = model(images)
            loss, _, _ = criterion(logits, masks)
            
        total_loss += loss.item()
        preds = logits.argmax(dim=1)
        
        for pred, mask in zip(preds, masks):
            ious = compute_iou(pred.cpu(), mask.cpu(), NUM_CLASSES)
            for cls, iou in enumerate(ious):
                if not np.isnan(iou):
                    all_ious[cls].append(iou)
                    
    class_ious = [np.mean(cls_ious) if cls_ious else 0.0 for cls_ious in all_ious]
    miou = np.mean([iou for iou in class_ious if iou > 0])
    return total_loss / len(loader), miou

# ── Kaggle Safety Wall & Checkpointing Setup ──
TIME_LIMIT_HOURS = 11.5
deadline = time.time() + TIME_LIMIT_HOURS * 3600
os.makedirs("/kaggle/working/checkpoints", exist_ok=True)

if "best_miou" not in locals(): best_miou = 0.0
history = {"train_loss": [], "val_loss": [], "miou": []}

# ── UI Setup ──
progress_html = display(HTML(f"<div><progress value='0' max='{NUM_EPOCHS}' style='width:300px; height:20px; vertical-align: middle;'></progress> [0/{NUM_EPOCHS}]</div>"), display_id=True)
table_html = display(HTML("<table border='1' class='dataframe'><thead><tr style='text-align: left;'><th>Epoch</th><th>Train Loss</th><th>Val Loss</th><th>mIoU</th><th>Time</th></tr></thead><tbody></tbody></table>"), display_id=True)
table_rows = ""

for epoch in range(start_epoch, NUM_EPOCHS):
    criterion.set_epoch(epoch) 
    epoch_start = time.time()
    if time.time() > deadline:
        print(f"\n⏰ 11.5 hour time budget reached! Saving and stopping gracefully.")
        break

    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, scaler, DEVICE)
    val_loss, miou = validate_one_epoch(model, val_loader, criterion, DEVICE)
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["miou"].append(miou)

    epoch_duration = time.time() - epoch_start
    epoch_times.append(epoch_duration)
    avg_epoch_time = sum(epoch_times) / len(epoch_times)
    eta_seconds = int(avg_epoch_time * (NUM_EPOCHS - (epoch + 1)))
    eta_str = str(datetime.timedelta(seconds=eta_seconds))
    
    # ── Update UI ──
    progress_html.update(HTML(f"<div><progress value='{epoch+1}' max='{NUM_EPOCHS}' style='width:300px; height:20px; vertical-align: middle;'></progress> [{epoch+1}/{NUM_EPOCHS} &lt; ETA: {eta_str}]</div>"))
    table_rows += f"<tr><td>{epoch+1}</td><td>{train_loss:.4f}</td><td>{val_loss:.4f}</td><td>{miou:.4f}</td><td>{epoch_duration:.1f}s</td></tr>"
    table_html.update(HTML(f"<table border='1' class='dataframe'><thead><tr style='text-align: left;'><th>Epoch</th><th>Train Loss</th><th>Val Loss</th><th>mIoU</th><th>Time</th></tr></thead><tbody>{table_rows}</tbody></table>"))

    # Save based on best mIoU
    if miou > best_miou:
        best_miou = miou
        torch.save(model.state_dict(), "/kaggle/working/best_model.pth")
        
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "best_miou": best_miou,
    }, f"/kaggle/working/checkpoints/unet_epoch_{epoch}.pth")

with open("/kaggle/working/training_history.json", "w") as f:
    json.dump(history, f, indent=2)


Downloading: "https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/efficientnet-b4-6ed6700e.pth" to /root/.cache/torch/hub/checkpoints/efficientnet-b4-6ed6700e.pth
100%|██████████| 74.4M/74.4M [00:00<00:00, 157MB/s]
/tmp/ipykernel_3071/748962305.py:24: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() # AMP Optimization


Epoch,Train Loss,Val Loss,mIoU,Time
1,0.4240,0.3103,0.3162,352.7s
2,0.2305,0.2080,0.3590,357.1s
3,0.1806,0.1706,0.3645,357.2s
4,0.1626,0.1646,0.3733,357.0s
5,0.1472,0.1511,0.3806,356.6s
6,0.1440,0.1548,0.3913,357.4s
7,0.1395,0.1503,0.3977,356.7s
8,0.1302,0.1440,0.3988,356.7s
9,0.1264,0.1433,0.4000,357.0s
10,0.1244,0.1389,0.3962,357.0s


Train:   0%|          | 0/363 [00:00<?, ?it/s]/tmp/ipykernel_3071/748962305.py:47: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Val:   0%|          | 0/41 [00:00<?, ?it/s]             /tmp/ipykernel_3071/748962305.py:69: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


In [17]:
# ── Cell X: Final Evaluation & Per-Class Metrics (mIoU / F1)
@torch.no_grad()
def compute_per_class_metrics(model, loader, num_classes, device):
    model.eval()
    intersection = torch.zeros(num_classes)
    union = torch.zeros(num_classes)
    tp = torch.zeros(num_classes)
    fp = torch.zeros(num_classes)
    fn = torch.zeros(num_classes)

    for images, masks in tqdm(loader, desc="Metrics"):
        images, masks = images.to(device), masks.to(device)
        logits = model(images)
        preds = torch.argmax(logits, dim=1)

        for c in range(num_classes):
            pred_c = (preds == c)
            true_c = (masks == c)
            intersection[c] += (pred_c & true_c).sum().item()
            union[c] += (pred_c | true_c).sum().item()
            tp[c] += (pred_c & true_c).sum().item()
            fp[c] += (pred_c & ~true_c).sum().item()
            fn[c] += (~pred_c & true_c).sum().item()

    iou = intersection / (union + 1e-8)
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)

    class_names = ["Background", "Intact", "Damaged", "Destroyed"]
    print(f"\n{'Class':<12} {'IoU':>8} {'Precision':>10} {'Recall':>8} {'F1':>8}")
    for c in range(num_classes):
        print(f"{class_names[c]:<12} {iou[c]:>8.4f} {precision[c]:>10.4f} {recall[c]:>8.4f} {f1[c]:>8.4f}")
    print(f"\nMean IoU: {iou.mean():.4f}")
    print(f"Mean F1:  {f1.mean():.4f}")

    return {"iou": iou, "precision": precision, "recall": recall, "f1": f1}

# Load best checkpoint before final eval
model.load_state_dict(torch.load("best_model.pth"))
metrics = compute_per_class_metrics(model, val_loader, NUM_CLASSES, DEVICE)

# At the end of Cell 9 (evaluation), add:
import csv
results = {
    "experiment_id": "MEMBER_X_EXP_Y",   # e.g., "B_arch_unetpp_r34"
    "variable": "architecture",
    "value": "UnetPlusPlus_resnet34",
    "miou": float(metrics["iou"].mean()),
    "bg_iou": float(metrics["iou"][0]),
    "intact_iou": float(metrics["iou"][1]),
    "damaged_iou": float(metrics["iou"][2]),
    "destroyed_iou": float(metrics["iou"][3]),
    "precision": float(metrics["precision"].mean()),
    "recall": float(metrics["recall"].mean()),
    "f1": float(metrics["f1"].mean()),
    "train_time_hrs": sum(epoch_times) / 3600,
    "notes": "",
}
with open(f"/kaggle/working/{results['experiment_id']}.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=results.keys())
    w.writeheader()
    w.writerow(results)

/tmp/ipykernel_3071/58024880.py:40: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_model.pth"))
Metrics: 100%|██████████| 41/41 [00:21<


Class             IoU  Precision   Recall       F1
Background     0.9509     0.9852   0.9647   0.9749
Intact         0.5441     0.6259   0.8063   0.7047
Damaged        0.3355     0.4843   0.5218   0.5024
Destroyed      0.2725     0.3894   0.4757   0.4283

Mean IoU: 0.5257
Mean F1:  0.6526


In [18]:
import pandas as pd
df = pd.read_csv("/kaggle/working/MEMBER_X_EXP_Y.csv")
display(df)

,experiment_id,variable,value,miou,bg_iou,intact_iou,damaged_iou,destroyed_iou,precision,recall,f1,train_time_hrs,notes
0,MEMBER_X_EXP_Y,architecture,UnetPlusPlus_resnet34,0.525743,0.950947,0.544101,0.335453,0.272472,0.621221,0.692137,0.65256,4.013285,NaN


In [ ]:
# ── Final Cell: Plot Training History & Push to HF Hub ─────────────────────
import json
import matplotlib.pyplot as plt
import os
import datetime
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

plot_path = "/kaggle/working/training_plot.png"

# 1. Generate and Save the Plot
if os.path.exists('/kaggle/working/training_history.json'):
    with open('/kaggle/working/training_history.json', 'r') as f:
        history = json.load(f)

    plt.figure(figsize=(10, 6))
    epochs = range(1, len(history['train_loss']) + 1)
    plt.plot(epochs, history['train_loss'], label='Train Loss', marker='o', color='blue')
    plt.plot(epochs, history['val_loss'], label='Validation Loss', marker='o', color='orange')

    plt.title('U-Net Training Progress (Combined CE + Dice Loss)')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    
    # Save the plot to the hard drive before showing it
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.show()
else:
    print('No training history found. Train the model first.')


# 2. Push Checkpoints, Model, and Plot to Hugging Face
HF_REPO_ID = "AbrarAlam/disasterm3-unet-checkpoints-2"

try:
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    api = HfApi()
    
    # Ensure the repo exists
    api.create_repo(repo_id=HF_REPO_ID, token=hf_token, exist_ok=True)
    
    # ── FIX: Point exactly to the hardcoded /kaggle/working/checkpoints folder ──
    actual_checkpoint_dir = "/kaggle/working/checkpoints"
    if os.path.exists(actual_checkpoint_dir):
        print("Uploading checkpoints folder...")
        api.upload_folder(
            folder_path=actual_checkpoint_dir,
            path_in_repo="checkpoints",    # <--- THIS KEEPS THEM IN A SUBFOLDER
            repo_id=HF_REPO_ID,
            token=hf_token,
            commit_message=f"Checkpoint at {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}",
        )

    
    # ── FIX: Point exactly to /kaggle/working/best_model.pth ──
    actual_best_path = "/kaggle/working/best_model.pth"
    if os.path.exists(actual_best_path):
        print("Uploading best_model.pth...")
        api.upload_file(
            path_or_fileobj=actual_best_path,
            path_in_repo="best_model.pth",
            repo_id=HF_REPO_ID,
            token=hf_token,
        )
    else:
        print(f"Skipping best model: {actual_best_path} not found.")

    # ── Upload the training plot ──
    if os.path.exists(plot_path):
        print("Uploading training plot...")
        api.upload_file(
            path_or_fileobj=plot_path,
            path_in_repo="training_plot.png",
            repo_id=HF_REPO_ID,
            token=hf_token,
        )
    
    print(f"\n✓ Successfully pushed Checkpoints, Model, and Training Plot to HF: {HF_REPO_ID}")
except Exception as e:
    print(f"\n⚠ HF push failed: {e}")
